
## Clasificación de Texto de las ODS


# 1. Importación de librerías requeridas

In [1]:
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk import RegexpTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report

# 2. Carga de datos

In [2]:
data = pd.read_excel("Train_textosODS.xlsx")
data.head()

,textos,ODS
0,"""Aprendizaje"" y ""educación"" se consideran sinó...",4
1,No dejar clara la naturaleza de estos riesgos ...,6
2,"Como resultado, un mayor y mejorado acceso al ...",13
3,Con el Congreso firmemente en control de la ju...,16
4,"Luego, dos secciones finales analizan las impl...",5


Se visualizan las diferentes categorías para ver qué tan desbalanceadas están las clases

In [3]:
data["ODS"].value_counts()

ODS
16    1080
5     1070
4     1025
3      894
7      787
6      695
11     607
1      505
13     464
8      446
14     377
2      369
10     352
9      343
15     330
12     312
Name: count, dtype: int64

Debido a que se utilizará XGBoost para la clasificación, éste espera etiquetas que comiencen desde 0 y no desde 1, se requiere convertir las etiquetas de la columna "ODS" para que sean de 0 a 15 y no de 1 a 16

In [4]:
data["ODS"] = data["ODS"] - data["ODS"].min()
data["ODS"].value_counts()

ODS
15    1080
4     1070
3     1025
2      894
6      787
5      695
10     607
0      505
12     464
7      446
13     377
1      369
9      352
8      343
14     330
11     312
Name: count, dtype: int64

Se evidencia que hay clases con mayor cantidad de elementos que otras clases, es un dataset desbalanceado, lo que se deberá tener en cuenta al momento de hacer el entrenamiento del modelo.

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9656 entries, 0 to 9655
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   textos  9656 non-null   object
 1   ODS     9656 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 151.0+ KB


In [6]:
data.duplicated().sum()

0

El conjunto de datos no contiene filas nulas ni duplicadas, por ende no se hace neceario eliminar ninguna fila.

## 2.1. División del dataset en entrenamiento y test

In [7]:
X_train, X_test, y_train, y_test = train_test_split(data["textos"], data["ODS"], test_size=0.2, random_state=52)

# 3. Creación de funciones para el Pipeline

Se utilizarán dos algoritmos de clustering para la creación de la paleta de colores: K-means y Mean Shift. Se utilizan estos dos modelos porque son ampliamnete utilizados para clusterización de imágenes. Además al ser una aplicación de generación de Paleta de colores los prototipos generados por los centros o centroides facilita el establecimiento de las paletas, pues los centroides representarían un conjunto de colores determinado.

## 3.1. Procesamiento de textos

Se crea un set con las palabras vacías del español ya que estás no agregan valor al momento de determinar el ODS, por ende en la función de preprocesamiento de texto no se tendrán en cuenta. Se utiliza un set en lugar de una lista para acelerar la búsqueda.

In [8]:
spanish_stopwords = set(stopwords.words('spanish'))

La función `text_preprocess` se utilizará para hacer el procesamiento de los textos.

1. `tokenizer = RegexpTokenizer(r'\w+')` realiza la tokenización por medio de expresiones regulares con la expresión `r'\w+'` que permite usar sólo caracteres alfanuméricos eliminando los signos de puntuación y caracteres especiales.
2. `stemmer = PorterStemmer()` reduce la palabra a su raíz gramatical`
3. `tokenize(text.lower())` permite hacer la tokenización con las palabras en minúscula


In [9]:
def text_preprocess(text):
    tokenizer = RegexpTokenizer(r'\w+')
    stemmer = PorterStemmer()
    tokens = tokenizer.tokenize(text.lower())
    tokens = [word for word in tokens if word not in spanish_stopwords]
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

## 3.2. Definición de pipelines para cada modelo

Se utilizarán tres pipelines para tres modelos de clasificación (XGBoost, SVM y Regresión logística), las pipelines contendrán los siguientes pasos:

1. Se usará `TfidfVectorizer` con el parámetro `preprocesor=text_preprocess` para convertir el texto procesado en valores numéricos. Al usar `TfidfVectorizer` se le da preponderancia a palabras que aparecen con menos frecuencia y que dan mayor significado al texto, mientra que se le da menos peso a palabras más comunes que pueden estar presentes en múltiples documentos. Se utliza `max_features=5000` para utilizar las 5000 palabras más relevantes reduciendo el tiempo de entrenamiento.
2. Se usará `TruncatedSVD` para hacer una reducción de dimensionalidad del vector obtenido en el paso anterior, este paso se requiere pues el vector generado en el paso uno es de alta dimensionalidad y `TruncatedSVD` permite mantener los componentes más importantes de una matriz dispersa (como la que tenemos en este caso, una matriz con muchos 0), la reducción de la dimensionalidad genera menores tiempos de entrenamiento y reduce el overfitting al eliminar el ruido de las columnas no aportantes manteniendo las relaciones principales de los datos. Se utiliza `n_components=100` para utilizar las 100 componentes principales resultantes luego de hacer la reducción de dimensionalidad, esto con el objetivo de reducir el tiempo de ejecución.
3. Aplicación del modelo supervisado para la clasificación (XGBoost, SVM y Regresión logística).

    3.1 `XGBoost`: se utilizará como métrica de evaluación "mlogloss" ya que es una métrica utilizada comunmente para aplicaciones de clasificación multiclase.

    3.2 `SVM`: se utiliza inicialmente un kernel lineal, sin embargo en la búsqueda de hiperparámetros se utilizarán varios. Se usa `probability=True` para hacer la comparación con los demás modelos.

    3.3 `Logistic Regression`: Se utiliza `max_iter=1000` para asegurarse que el modelo llegue a converger.



In [10]:
pipelines = {
    "XGBoost": Pipeline([
        ("tfidf", TfidfVectorizer(preprocessor=text_preprocess, max_features=5000)), 
        ("svd", TruncatedSVD(n_components=100)),  
        ("classifier", XGBClassifier(eval_metric="mlogloss"))  
    ]),
    "SVM": Pipeline([
        ("tfidf", TfidfVectorizer(preprocessor=text_preprocess, max_features=5000)),
        ("svd", TruncatedSVD(n_components=100)),
        ("classifier", SVC(kernel="linear", probability=True))
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(preprocessor=text_preprocess, max_features=5000)),
        ("svd", TruncatedSVD(n_components=100)),
        ("classifier", LogisticRegression(max_iter=1000))
    ])
}

## 3.3. Definición de búsqueda de hiperparámetros

Para cada pipeline se realizará una búsqueda aleatoria de hiperparámetros `RandomizedSearchCV` instanciados en el diccionario `param_grids` así:

1. `XGBoost`: se definen el número de árboles para que tome valores de 50, 100 y 200, con profundidades de árbol de 3, 5, y 7 y una tasa de aprendizaje 0.01, 0.1 y 0.2.
2. `SVM`: se busca el hiperparámetro de regularización C que controla la complejidad del modelo se toman valores desde 0.1 para mayor regularización hasta 10 para menor regularización. Se utilizan Kernels lineales y rbf para evaluar cuál se ajusta mejor a los datos, rbf permite obtner límites de decisión no lineales.
3. `Logistic Regression`: se utilizan dos solvers `lbfgs` y `liblinear`, para evaluar los mejores pesos de coeficientes para cada modelo y al igual que en SVM se utilizan distintos hiperparámetros para controlar la regularización.

In [11]:
param_grids = {
    "XGBoost": {
        "classifier__n_estimators": [50, 100, 200],
        "classifier__max_depth": [3, 5, 7],
        "classifier__learning_rate": [0.01, 0.1, 0.2]
    },
    "SVM": {
        "classifier__C": [0.1, 1, 10],
        "classifier__kernel": ["linear", "rbf"]
    },
    "Logistic Regression": {
        "classifier__C": [0.1, 1, 10],
        "classifier__solver": ["lbfgs", "liblinear"]
    }
}

## 3.4. Función para visualización de resultados

Se crea la función `report_best_scores` y `see_results` para ver el ranking de los modelos para cada tipo de clasificador y ver cómo actúan los distintos hiperparámetros en cada modelo.

In [12]:
def report_best_scores(results, n_top=3):
    # Esta función espera una instancia de resultados de búsqueda de cross validation, por ejemplo: search.cv_results_
    for i in range(1, n_top + 1):
        candidates = np.flatnonzero(results['rank_test_score'] == i)
        for candidate in candidates:
            print("Model with rank: {0}".format(i))
            print("Mean validation score: {0:.3f} (std: {1:.3f})".format(
                  results['mean_test_score'][candidate],
                  results['std_test_score'][candidate]))
            print("Parameters: {0}".format(results['params'][candidate]))
            print("")
    # Retorna los parámetros del mejor modelo basado en accuracy.
    return list(results.sort_values("rank_test_score")['params'])[0]

def see_results(results):
    # Esta función espera una instancia de pandas dataframe de los resultados de búsqueda de cross validation, por ejemplo: pd.DataFrame(search.cv_results_)
    display(results[results.columns.drop(list(results.filter(regex='split')))].sort_values("rank_test_score"))

# 4. Selección del mejor modelo

Para la selección del modelo se utilizarán máximo 10 iteraciones mediante la clase `RandomizedSearchCV` para buscar los mejores hiperparámetros de cada modelo, para crear los conjuntos de validación se hará una partición de 5 splits utilizando la clase `StratifiedKFold` que permite asegurar que todas las categorías estén presentes en los splits y que no se le de preponderancia a una categoría sobre otra y que esto pueda afectar los resultados.

Finalmente se escoge el mejor modelo y se visualizan las métricas de evaluación por medio de la clase `classification_report` luego de hacer las predicciones con el dataset de test. Se utiliza `Accuracy` como medida de evaluación para seleccionar el mejor modelo ya que es una métrica que sirve para evaluar clasificacione multiclase.

In [13]:
best_model = None
best_score = 0
best_pipeline = None

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=52)

for model_name, pipeline in pipelines.items():

    print(f"Entrenando modelo: {model_name}")

    search = RandomizedSearchCV(
        pipeline, param_grids[model_name], n_iter=10, cv=kfold,
        scoring='accuracy', random_state=52, n_jobs=-1, return_train_score=True
    )
    search.fit(X_train, y_train)
    
    results_df = pd.DataFrame(search.cv_results_)
    see_results(results_df)
    best_params = report_best_scores(results_df)

   
    if search.best_score_ > best_score:
        best_model = model_name
        best_score = search.best_score_
        best_pipeline = search.best_estimator_


print(f"Mejor modelo: {best_model} Accuracy {best_score:.4f}")
y_pred = best_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

Entrenando modelo: XGBoost


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__n_estimators,param_classifier__max_depth,param_classifier__learning_rate,params,mean_test_score,std_test_score,rank_test_score,mean_train_score,std_train_score
3,45.352578,0.276270,2.043605,0.042710,200,5,0.1,"{'classifier__n_estimators': 200, 'classifier_...",0.840110,0.007791,1,1.000000,0.000000
9,16.627145,1.615944,1.517875,0.385065,100,3,0.2,"{'classifier__n_estimators': 100, 'classifier_...",0.836874,0.008542,2,0.999482,0.000189
2,13.105015,0.255526,1.819588,0.053031,50,3,0.2,"{'classifier__n_estimators': 50, 'classifier__...",0.826257,0.008994,3,0.957373,0.001763
5,21.129252,0.359844,2.090075,0.083443,50,5,0.1,"{'classifier__n_estimators': 50, 'classifier__...",0.817583,0.009163,4,0.984755,0.001423
1,12.604760,0.338941,1.700558,0.091719,50,3,0.1,"{'classifier__n_estimators': 50, 'classifier__...",0.811110,0.009633,5,0.890860,0.001795
7,107.033541,2.485102,1.880825,0.362517,200,7,0.01,"{'classifier__n_estimators': 200, 'classifier_...",0.807225,0.005885,6,0.978250,0.001115
0,52.517370,1.085179,1.971864,0.070034,100,7,0.01,"{'classifier__n_estimators': 100, 'classifier_...",0.798033,0.007470,7,0.954460,0.002003
4,34.400587,0.526963,1.992708,0.045896,100,5,0.01,"{'classifier__n_estimators': 100, 'classifier_...",0.790912,0.005707,8,0.898401,0.003583
8,32.599410,0.270841,2.004896,0.065870,50,7,0.01,"{'classifier__n_estimators': 50, 'classifier__...",0.787416,0.004796,9,0.933616,0.001630
6,29.832820,0.157179,1.962898,0.065799,200,3,0.01,"{'classifier__n_estimators': 200, 'classifier_...",0.784439,0.007939,10,0.839073,0.002152


Model with rank: 1
Mean validation score: 0.840 (std: 0.008)
Parameters: {'classifier__n_estimators': 200, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.1}

Model with rank: 2
Mean validation score: 0.837 (std: 0.009)
Parameters: {'classifier__n_estimators': 100, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.2}

Model with rank: 3
Mean validation score: 0.826 (std: 0.009)
Parameters: {'classifier__n_estimators': 50, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.2}

Entrenando modelo: SVM


/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_search.py:318: UserWarning: The total space of parameters 6 is smaller than n_iter=10. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__kernel,param_classifier__C,params,mean_test_score,std_test_score,rank_test_score,mean_train_score,std_train_score
3,25.655554,0.268974,3.524959,0.061020,rbf,1,"{'classifier__kernel': 'rbf', 'classifier__C': 1}",0.857718,0.008024,1,0.944653,0.001073
5,21.492380,1.038795,2.688029,0.025224,rbf,10,"{'classifier__kernel': 'rbf', 'classifier__C':...",0.853833,0.007676,2,0.999612,0.000220
4,15.543354,0.220635,2.308465,0.153566,linear,10,"{'classifier__kernel': 'linear', 'classifier__...",0.853704,0.006120,3,0.914002,0.001464
2,20.998537,0.132716,2.526204,0.058426,linear,1,"{'classifier__kernel': 'linear', 'classifier__...",0.846453,0.008118,4,0.870469,0.002599
1,41.249551,0.551536,3.935152,0.101726,rbf,0.1,"{'classifier__kernel': 'rbf', 'classifier__C':...",0.826645,0.006817,5,0.847747,0.001542
0,35.694837,0.259661,2.933839,0.065821,linear,0.1,"{'classifier__kernel': 'linear', 'classifier__...",0.710643,0.006787,6,0.727182,0.003063


Model with rank: 1
Mean validation score: 0.858 (std: 0.008)
Parameters: {'classifier__kernel': 'rbf', 'classifier__C': 1}

Model with rank: 2
Mean validation score: 0.854 (std: 0.008)
Parameters: {'classifier__kernel': 'rbf', 'classifier__C': 10}

Model with rank: 3
Mean validation score: 0.854 (std: 0.006)
Parameters: {'classifier__kernel': 'linear', 'classifier__C': 10}

Entrenando modelo: Logistic Regression


/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_search.py:318: UserWarning: The total space of parameters 6 is smaller than n_iter=10. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__solver,param_classifier__C,params,mean_test_score,std_test_score,rank_test_score,mean_train_score,std_train_score
4,8.838962,0.519709,2.031166,0.239129,lbfgs,10,"{'classifier__solver': 'lbfgs', 'classifier__C...",0.858365,0.008678,1,0.891733,0.002634
5,10.239791,0.086007,1.803694,0.031411,liblinear,10,"{'classifier__solver': 'liblinear', 'classifie...",0.853703,0.005512,2,0.883286,0.002320
2,8.960212,0.122061,2.078434,0.057097,lbfgs,1,"{'classifier__solver': 'lbfgs', 'classifier__C...",0.842051,0.005361,3,0.859885,0.002420
3,10.999035,0.329097,2.125032,0.070637,liblinear,1,"{'classifier__solver': 'liblinear', 'classifie...",0.839204,0.007851,4,0.854965,0.002213
0,7.837054,0.168193,1.991457,0.032012,lbfgs,0.1,"{'classifier__solver': 'lbfgs', 'classifier__C...",0.683973,0.006247,5,0.698764,0.003166
1,9.645656,0.508413,2.044353,0.098228,liblinear,0.1,"{'classifier__solver': 'liblinear', 'classifie...",0.680478,0.006317,6,0.695365,0.002543


Model with rank: 1
Mean validation score: 0.858 (std: 0.009)
Parameters: {'classifier__solver': 'lbfgs', 'classifier__C': 10}

Model with rank: 2
Mean validation score: 0.854 (std: 0.006)
Parameters: {'classifier__solver': 'liblinear', 'classifier__C': 10}

Model with rank: 3
Mean validation score: 0.842 (std: 0.005)
Parameters: {'classifier__solver': 'lbfgs', 'classifier__C': 1}

Mejor modelo: Logistic Regression Accuracy 0.8584
              precision    recall  f1-score   support

           0       0.88      0.81      0.84       100
           1       0.80      0.77      0.78        66
           2       0.85      0.92      0.88       167
           3       0.94      0.93      0.94       221
           4       0.91      0.96      0.93       220
           5       0.96      0.88      0.92       156
           6       0.85      0.86      0.86       154
           7       0.72      0.71      0.71       102
           8       0.65      0.68      0.66        62
           9       0.64  

# 5. Conclusiones

1. Se obtuvo como mejor modelo el de regresión logística con el optimizador `lbfgs` el cual es recomendado para clasificaciones multiclase y con una parámetro de regularización C=10, el cual al ser un valor alto produce menor penalidad en la regularización y permite límites de decisión más complejos y capturar relaciones más complejas en el dataset.
2. El modelo seleccionado tiene buenas métricas para las diferentes categorías de ODS, sin embargo, se tienen valores más bajos para las categorías 8, 9 y 11, al modelo se le dificulta más clasificar estas categorías de manera correcta, esto quizá es debido al desbalanceo que se presenta en estas clases o también se puede presentar porque estas clases sean similares a otras, las métricas de recall son las más bajas por ende se generan más falsos positivos en estas categorías.
3. Las métricas muestran que no se genera un overfitting en el modelo pues hay una distribución consistente del f1-score en todas las clases, lo que indica una buena generalización del modelo.
4. El modelo es especialmente bueno en las categorías 3, 5, 15 y 14 donde se presentan altos valores de precisión y recall.
5. Para mejorar el desempeño del modelo para las clases con recall bajo se podría hacer un balanceo de clases por medio de oversampling con métodos como SMOTE.
6. Si bien se obtuvo como mejor modelo el de regresión logística, los modelos de XGBoost y SVM también obtuvieron buenas métricas de Accuracy, aunque con un tiempo de entrenamiento mayor, dependiendo de la aplicación también se podría considerar usar estos algoritmos.
7. El rol de utilizar TF-IDF y SVD en este tipo de escenarios es primordial pues se optimiza el tiempo de entrenamiento y el desempeño del modelo, TF-IDF permite filtrar las palabras qué más valor aportan para la clasificación mientras que SVM mejorar la eficiencia del modelo al trbajar con una dimensionalidad menor y con menos data inaportante que lleva a reducir el overfitting.
